# 12 — The Capstone: Did That Change Actually Help?

*(Written and reasoned through carefully; not yet run live in this session -- same daily token cap as notebook 11. Verify the printed numbers, don't assume them.)*

Eleven notebooks built pieces: retrieval metrics, context relevance, faithfulness, correctness, automated judges calibrated against your own manual verdicts, and your own synthetic eval set. This one uses all of it for the reason evaluation exists in the first place -- back in notebook 0 of the theory phase: *"How can I prove it?"*

The change under test: every answer in this series so far, from notebook 00 onward, used one plain prompt -- "answer using only this context." What happens if that prompt explicitly tells the model what to say when the context doesn't cover the question, instead of leaving it to guess at its own phrasing?


In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-20b"

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)


## The two versions

Version A is the prompt every notebook in this series has used so far -- it's already baked into `examples[i]["system_answer"]` from notebook 00, no need to regenerate it. Version B adds one explicit instruction: what exact phrase to use when the context doesn't cover the question.


In [ ]:
PROMPT_A = "Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"

PROMPT_B = (
    "Answer using ONLY this context. If the context does not contain the answer, "
    "respond with exactly: \"Not stated in the documents.\" Do not guess or partially answer "
    "in that case.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)


def generate_answer(question: str, context: str, prompt_template: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt_template.format(context=context, question=question)}],
        temperature=0.1,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


# Version A already exists in the dataset (every notebook so far used this exact prompt shape).
# Regenerate Version B for all 20 examples.
version_b_answers = [
    generate_answer(ex["question"], ex["context_text"], PROMPT_B) for ex in examples
]

print("Regenerated 20 answers under Version B.")
print("Example [16] -- Version A:", examples[16]["system_answer"])
print("Example [16] -- Version B:", version_b_answers[16])


## Re-running the full metric suite, both versions

Same `FaithfulnessMetric` and `CorrectnessMetric` from notebook 08 -- no new metrics, no new judge. The whole point of building them once, carefully, is reusing them unchanged here.


In [ ]:
FAITHFULNESS_PROMPT = """You are a faithfulness judge. Extract every factual claim in the \
answer below, then mark each as "supported" or "unsupported" based only on the context. \
Return strict JSON only, no other text, in this shape:
{{"claims": [{{"text": "...", "supported": true, "reason": "..."}}]}}

Context:
{context}

Answer:
{answer}"""

CORRECTNESS_PROMPT = """Expected answer: {gold}
Actual answer: {answer}

Is the actual answer correct and equivalent to the expected answer? Consider partial matches -- \
an answer that supports a real but different clause than the expected one should be scored \
PARTIALLY_CORRECT, not INCORRECT. If the expected answer is "Not stated in the documents" and \
the actual answer honestly declines to answer, that counts as CORRECT.

Return strict JSON only: {{"verdict": "CORRECT|PARTIALLY_CORRECT|INCORRECT", "reason": "..."}}"""

VERDICT_SCORES = {"CORRECT": 1.0, "PARTIALLY_CORRECT": 0.5, "INCORRECT": 0.0}


class FaithfulnessMetric:
    def __init__(self, client, model=MODEL):
        self.client, self.model = client, model

    def score(self, context: str, answer: str) -> float:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": FAITHFULNESS_PROMPT.format(context=context, answer=answer)}],
            temperature=0.0,
            reasoning_effort="low",
        )
        claims = json.loads(response.choices[0].message.content)["claims"]
        return 1.0 if not claims else sum(1 for c in claims if c["supported"]) / len(claims)


class CorrectnessMetric:
    def __init__(self, client, model=MODEL):
        self.client, self.model = client, model

    def score(self, gold: str, answer: str) -> float:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": CORRECTNESS_PROMPT.format(gold=gold, answer=answer)}],
            temperature=0.0,
            reasoning_effort="low",
        )
        verdict = json.loads(response.choices[0].message.content)["verdict"]
        return VERDICT_SCORES[verdict]


faithfulness_metric = FaithfulnessMetric(client)
correctness_metric = CorrectnessMetric(client)


def run_eval_suite(answers: list[str]) -> list[dict]:
    results = []
    for ex, answer in zip(examples, answers):
        f_score = faithfulness_metric.score(ex["context_text"], answer)
        c_score = correctness_metric.score(ex["gold_answer"], answer)
        results.append({"source": ex["source"], "faithfulness": f_score, "correctness": c_score})
    return results


results_a = run_eval_suite([ex["system_answer"] for ex in examples])
results_b = run_eval_suite(version_b_answers)


## The before/after comparison

Overall averages first, then the subset the change actually targeted -- the 5 unanswerable questions (indices 15-19) -- since that's where Version B's explicit instruction should matter most, if it matters at all.


In [ ]:
def avg(results: list[dict], key: str) -> float:
    return sum(r[key] for r in results) / len(results)


print(f"{'Metric':<15}{'Version A':<12}{'Version B':<12}{'Change'}")
for key in ("faithfulness", "correctness"):
    a, b = avg(results_a, key), avg(results_b, key)
    print(f"{key:<15}{a:<12.2f}{b:<12.2f}{b - a:+.2f}")

print()
print("Unanswerable subset only (indices 15-19):")
unans_a, unans_b = results_a[15:20], results_b[15:20]
for key in ("faithfulness", "correctness"):
    a, b = avg(unans_a, key), avg(unans_b, key)
    print(f"  {key:<13}{a:<12.2f}{b:<12.2f}{b - a:+.2f}")


## Reading the result honestly

Three possible outcomes, and each one means something different:

- **Correctness on the unanswerable subset went up, overall faithfulness held steady.** The change worked exactly as intended -- a more precise fallback instruction produced more of the exact refusal phrasing the gold labels expected, without making the model more evasive on questions it *could* actually answer.
- **Nothing moved.** The model was already handling unanswerable questions reasonably well under Version A (notebook 00 already showed honest refusals were common), so a more explicit instruction had nothing left to fix. Not a failed experiment -- a real, useful finding: this particular lever wasn't the bottleneck.
- **Correctness went up on the unanswerable subset, but faithfulness or correctness dropped elsewhere.** The stricter fallback instruction made the model *too* eager to decline, refusing on borderline-but-actually-answerable questions too. That's the regression a before/after comparison exists to catch -- and it's exactly the kind of tradeoff a single "quality went up" headline number would hide.

Whichever of these actually happened when you ran this: that's the answer to "did it help," backed by the same metrics you built and calibrated across the rest of this series -- not a guess, and not a vibe.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Regression suite | The full set of metrics, re-run after a change, to check nothing got worse while something else got better |
| Before/after comparison | Measuring the same eval set under both the old and new version of a pipeline component |

**This closes the practical series.** Notebook 0 (theory) opened with: *"Was retrieval the problem? Was chunking the problem? Was the LLM the problem? How can I prove it?"* Twelve notebooks later, you have real, specific tools for every one of those questions -- and, just as importantly, you've watched several of those tools disagree with each other, fail in reproducible and non-reproducible ways, and need real calibration against your own judgment before you'd trust them on anything you hadn't read yourself. That's a genuinely different, more useful place to stand than knowing the names of three eval frameworks.

**Exercises:**

- Try a third prompt version -- one that also explicitly encourages the model to quote the exact supporting sentence in its answer. Re-run the full suite: does citation-style answering change faithfulness, correctness, or both?
- Look specifically at whether Version B changed anything on the CUAD examples (indices 0-9), which were never the target of this change. Any unexpected movement there is worth investigating the same way notebook 02 investigated an unexpected root cause.
- This capstone only compared two prompts. Design (you don't have to run it) what a genuine retrieval-side regression test would need: what would you change, and which of notebooks 03's or 04's metrics would actually catch a regression there?
